# CHCEŠ MYŠ? · 02 · ILUSTRACE KOCOUR

Samostatný obrazový běh pro kartu **Kocour**. Vstupní obrázek se načítá přímo z repozitáře `artmisanovak-prog/chces-mys`, cesta `doma/ilustrace/kocour.jpg`.

Tento notebook provádí pouze obrazovou fázi. Nepřidává pohádku ani samostatný textový vstup. Model je **Qwen3-VL-Embedding-2B**. Výstupem je embeddingový vektor a archiv s podklady a metadaty, nikoli slovní interpretace obrázku.

## 1. Připojení Google Drive

Výsledky se průběžně ukládají na Google Drive, aby se neztratily při přerušení relace. Po dokončení se vytvoří ZIP pro stažení do telefonu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, time, zipfile, shutil, hashlib, platform
from datetime import datetime, timezone

PROJECT_DIR = Path('/content/CHCES_MYS_02_KOCOUR')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR = Path('/content/drive/MyDrive/CHCES_MYS_EXPERIMENTY/02_KOCOUR_IMAGE')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

print("Pracovní složka:", PROJECT_DIR)
print("Průběžné ukládání:", DRIVE_DIR)

## 2. Stažení zdrojového obrázku

Zdroj je jiný repozitář než v předchozím notebooku Myš: `chces-mys`. Obrázek se používá přímo v orientaci, v jaké je uložen v tomto repozitáři. Notebook jej neotáčí ani jinak neupravuje.

In [ ]:
import requests
from PIL import Image
from IPython.display import display

IMAGE_URL = "https://raw.githubusercontent.com/artmisanovak-prog/chces-mys/main/doma/ilustrace/kocour.jpg"
IMAGE_PATH = PROJECT_DIR / "kocour.jpg"

response = requests.get(IMAGE_URL, timeout=60)
response.raise_for_status()
IMAGE_PATH.write_bytes(response.content)

img = Image.open(IMAGE_PATH).convert("RGB")
display(img)
print("Soubor:", IMAGE_PATH.name)
print("Rozměry:", img.size)
print("SHA-256:", hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest())

## 3. Experimentální instrukce

Instrukce jsou stejné jako v obrazovém běhu Myš. Mění se pouze obrazový vstup: místo `mys.jpg` je zde `kocour.jpg`. Text viditelný přímo v ilustraci zůstává součástí obrazového artefaktu. Není to samostatný textový/pohádkový běh.

In [ ]:
INSTRUCTION = """Represent the visual artifact from the project "Chceš myš?" according to the
following project-specific conceptual framework. This is an image-only task.
Do not invent or import a story, a plot, or any text that was not supplied as
part of the image itself.

Within the illustration's conceptual system:
- The girl represents speech.
- The animal represents a disturbance or disruption of that speech.
- Their relation concerns language as represented through the visual artifact.
- The gap is not a mistake, missing information, or a simple mismatch.
- The gap is an active space in which interpretation can take place.
- This interpretive space can be approached from several points of view.
- These points of view have a circular topology, with relations such as
  adjacency and opposition. Do not assume or invent a fixed number of points
  of view, names, coordinates, or a complete topology when they have not
  been supplied.

Treat the handwritten wording visible in the illustration as part of the
visual artifact. Do not treat it as separate story text. Do not reduce the
task to identifying objects, matching labels, or deciding which other image
or text this illustration resembles.

Create a semantic representation of this illustration under these
instructions. Preserve ambiguity where the image does not determine a
single interpretation. Do not add a verbal explanation as a substitute
for the embedding."""

INSTRUCTION_PATH = PROJECT_DIR / "instructions.txt"
INSTRUCTION_PATH.write_text(INSTRUCTION, encoding="utf-8")
print(INSTRUCTION)
print("\nPočet znaků instrukce:", len(INSTRUCTION))

## 4. Instalace oficiální implementace Qwen

Používá se oficiální repozitář Qwen3-VL-Embedding. Instalace a načtení modelu proběhnou až v tomto kroku.

In [ ]:
import subprocess, sys, os

QWEN_REPO = Path('/content/Qwen3-VL-Embedding')
if not QWEN_REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/QwenLM/Qwen3-VL-Embedding.git", str(QWEN_REPO)],
        check=True
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(QWEN_REPO)],
    check=True
)
print("Oficiální repozitář připraven:", QWEN_REPO)

## 5. Automatická volba zařízení a načtení modelu

Notebook nic nezamyká na CPU ani GPU. Použije CUDA, pokud ji runtime zpřístupní, jinak CPU. Samotné načtení modelu a výpočet embeddingu jsou nejnáročnější části běhu.

In [ ]:
import torch
from src.models.qwen3_vl_embedding import Qwen3VLEmbedder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Zvolené zařízení:", DEVICE)
print("CUDA dostupná:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "Qwen/Qwen3-VL-Embedding-2B"
model = Qwen3VLEmbedder(model_name_or_path=MODEL_NAME)
print("Model načten:", MODEL_NAME)

## 6. Výpočet obrazové reprezentace

Do modelu vstupuje jediný obrázek a výše uvedená projektová instrukce. Nevkládá se pohádka ani jiný textový dokument.

In [ ]:
run_started = datetime.now(timezone.utc).isoformat()
log_path = PROJECT_DIR / "run_log.jsonl"

def log_event(event, **kwargs):
    record = {"timestamp": datetime.now(timezone.utc).isoformat(), "event": event, **kwargs}
    with log_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    shutil.copy2(log_path, DRIVE_DIR / "run_log.jsonl")

log_event("image_loaded", image="kocour.jpg", image_url=IMAGE_URL, dimensions=list(img.size))
log_event("model_loading_complete", model=MODEL_NAME, device=DEVICE)

inputs = [{
    "image": str(IMAGE_PATH),
    "instruction": INSTRUCTION
}]
embeddings = model.process(inputs)

if hasattr(embeddings, "detach"):
    embedding = embeddings.detach().cpu().numpy()
else:
    import numpy as np
    embedding = np.asarray(embeddings)

import numpy as np
embedding = np.asarray(embedding, dtype=np.float32)
embedding_path = PROJECT_DIR / "kocour_image_embedding.npy"
np.save(embedding_path, embedding)

print("Tvar embeddingu:", embedding.shape)
print("Datový typ:", embedding.dtype)
print("Norma:", float(np.linalg.norm(embedding)))
log_event("embedding_saved", shape=list(embedding.shape), dtype=str(embedding.dtype),
          norm=float(np.linalg.norm(embedding)))

## 7. Metadata a průběžné uložení

Metadata zaznamenávají zdroj, přesnou instrukci, verzi knihoven, zařízení a parametry výsledného vektoru. Nezaznamenává se žádné ruční otáčení ani transformace obrázku, protože se zde žádná neprovádí.

In [ ]:
metadata = {
    "project": "CHCEŠ MYŠ?",
    "experiment_id": "02",
    "phase": "image_only",
    "card": "kocour",
    "model": MODEL_NAME,
    "source_repository": "artmisanovak-prog/chces-mys",
    "source_path": "doma/ilustrace/kocour.jpg",
    "source_url": IMAGE_URL,
    "source_sha256": hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest(),
    "image_dimensions": list(img.size),
    "instruction": INSTRUCTION,
    "embedding_shape": list(embedding.shape),
    "embedding_dtype": str(embedding.dtype),
    "embedding_norm": float(np.linalg.norm(embedding)),
    "device": DEVICE,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "cuda_available": torch.cuda.is_available(),
    "run_started_utc": run_started,
    "run_finished_utc": datetime.now(timezone.utc).isoformat()
}
metadata_path = PROJECT_DIR / "metadata.json"
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

for p in [IMAGE_PATH, INSTRUCTION_PATH, embedding_path, metadata_path, log_path]:
    if p.exists():
        shutil.copy2(p, DRIVE_DIR / p.name)

print("Uloženo na Google Drive:", DRIVE_DIR)
print("Obsah:", [p.name for p in DRIVE_DIR.iterdir()])

## 8. Vytvoření ZIP archivu

Archiv obsahuje původní vstupní obrázek, přesnou instrukci, metadata, log běhu a embedding. Po vytvoření se uloží na Google Drive i do pracovní složky Colabu.

In [ ]:
ZIP_NAME = "CHCES_MYS_KOCOUR_IMAGE_02.zip"
ZIP_PATH = Path('/content') / ZIP_NAME
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for filename in ["kocour.jpg", "instructions.txt", "metadata.json",
                     "kocour_image_embedding.npy", "run_log.jsonl"]:
        p = PROJECT_DIR / filename
        if p.exists():
            z.write(p, arcname=filename)

DRIVE_ZIP = DRIVE_DIR / ZIP_NAME
shutil.copy2(ZIP_PATH, DRIVE_ZIP)
print("ZIP vytvořen:", ZIP_PATH)
print("Kopie na Drive:", DRIVE_ZIP)
print("Velikost ZIP (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 3))

## 9. Stažení ZIP do zařízení

Spusť tuto buňku a Colab nabídne stažení archivu. Původní kopie zůstává také na Google Drive.

In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))

## Poznámka k interpretaci výsledku

Embedding je číselná reprezentace vytvořená modelem podle zadané instrukce. Sám o sobě není slovním vysvětlením ani důkazem, že model správně pochopil jednotlivé pojmy projektu. Interpretace a srovnání přijdou až při následné analýze uložených výsledků.